In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import pandas as pd
import numpy as np

train = pd.read_pickle('data/train_processed.pkl')
test = pd.read_pickle('data/test_processed.pkl')

Mounted at /content/drive


In [ ]:
# ============================================================
# MLflow / DagsHub experiment tracking
# ============================================================
!pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='skupr23', repo_name='ml_final', mlflow=True)

import mlflow

EXPERIMENT_NAME = 'LightGBM_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)

# Close any run left open by a previous (interrupted) execution of this notebook.
if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)

# Cell 2 - WMAE metric, the actual competition scoring function


In [2]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# Cell 3 - Chronological validation split


In [13]:
train = train.sort_values('Date')
val = train[train['Date'] >= train['Date'].max() - pd.Timedelta(weeks=39)].copy()
fit = train[train['Date'] < val['Date'].min()].copy()
print('fit ends:', fit['Date'].max())
print('val range:', val['Date'].min(), 'to', val['Date'].max())

fit ends: 2012-01-20 00:00:00
val range: 2012-01-27 00:00:00 to 2012-10-26 00:00:00


# Cell 4 - Baseline: 52-week seasonal naive


In [14]:
# This is the number every real model MUST beat. If LightGBM can't beat
# "just predict what happened last year", something is wrong.
baseline_pred = val['lag_52'].fillna(val['storedept_median_sales'])
baseline_score = wmae(val['Weekly_Sales'], baseline_pred, val['IsHoliday'])
print('52-week seasonal baseline WMAE:', baseline_score)

52-week seasonal baseline WMAE: 1789.9133525504185


# Cell 5 - Prepare ALL features (no selection yet) for LightGBM


In [15]:
cat_cols = ['Store','Dept','Type','holiday_type']
for df in [fit, val, test]:
    for c in cat_cols:
        df[c] = df[c].astype('category')

drop_cols = ['Weekly_Sales','Date','Id'] if 'Id' in fit.columns else ['Weekly_Sales','Date']
feature_cols = [c for c in fit.columns if c not in drop_cols]

X_fit, y_fit = fit[feature_cols], fit['Weekly_Sales']
X_val, y_val = val[feature_cols], val['Weekly_Sales']
print('Total features going in:', len(feature_cols))

Total features going in: 52


# Cell 6 - Train LightGBM on ALL features (baseline model run)

In [16]:
import lightgbm as lgb

model_all = lgb.LGBMRegressor(
    objective='mae',       # MAE loss aligns directly with WMAE scoring
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=64,
    random_state=42
)
model_all.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.247033 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8042
[LightGBM] [Info] Number of data points in the train set: 303030, number of used features: 45
[LightGBM] [Info] Start training from score 7679.899902
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 2115.14
[200]	valid_0's l1: 1374.13
[300]	valid_0's l1: 1322.38
[400]	valid_0's l1: 1308.7
[500]	valid_0's l1: 1293.68
[600]	valid_0's l1: 1287.6
[700]	valid_0's l1: 1285.6
[800]	valid_0's l1: 1282.14
[900]	valid_0's l1: 1280.35
[1000]	valid_0's l1: 1278.9
[1100]	valid_0's l1: 1277.78
[1200]	valid_0's l1: 1276.2
[1300]	valid_0's l1: 1275.25
[1400]	valid_0's l1: 1274.32
[1500]	valid_0's l1: 1272.56
[1600]	valid_0's l1: 1271.74
Early stopping, best iteration is:
[1581]	valid_0's l1: 1271.63


LGBMRegressor(learning_rate=0.03, n_estimators=2000, num_leaves=64,
              objective='mae', random_state=42)

# Cell 7 - Evaluate the all-features model


In [17]:
val_pred_all = model_all.predict(X_val)
wmae_all = wmae(y_val, val_pred_all, val['IsHoliday'])
print('LightGBM (all features) WMAE:', wmae_all)
print('vs baseline:', baseline_score)

LightGBM (all features) WMAE: 1301.4343028455785
vs baseline: 1789.9133525504185


# Cell 8 - Feature importance -> this IS our feature selection step


In [18]:
# LightGBM tells us which features it actually split on. Features with
# ~0 importance are just noise/dead weight - we drop them.
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_all.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

                    feature  importance
1                      Dept       18946
0                     Store       14421
30                    lag_1        5595
27             week_of_year        4715
46           sales_change_1        4464
51   storedept_median_sales        4062
36                   lag_52        3717
3               Temperature        3445
4                Fuel_Price        3184
32                    lag_4        2994
28                 week_sin        2927
31                    lag_2        2859
33                   lag_13        2508
29                 week_cos        2499
34                   lag_26        2187
47     sales_per_store_size        1994
38         rolling_mean_4_x        1830
37                   lag_53        1732
35                   lag_51        1718
39       rolling_median_8_x        1566
10                      CPI        1538
41        rolling_mean_26_x        1321
40        rolling_mean_13_x        1279
26                    month        1256


# Cell 9 - Select top features


In [19]:
# Reasoning: keep only features that contributed meaningfully to splits.
# We're not being too aggressive (trees tolerate extra features fine),
# just cutting dead weight - e.g. markdown missing-flags for markdowns
# the model never split on, or redundant lags.
selected_features = importances[importances['importance'] > 0]['feature'].tolist()
print(f'Dropped {len(feature_cols) - len(selected_features)} zero-importance features')
print('Kept:', len(selected_features))

Dropped 12 zero-importance features
Kept: 40


# Cell 10 - Retrain on selected features only


In [20]:
X_fit_sel = fit[selected_features]
X_val_sel = val[selected_features]

model_sel = lgb.LGBMRegressor(
    objective='mae',
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=64,
    random_state=42
)
model_sel.fit(
    X_fit_sel, y_fit,
    eval_set=[(X_val_sel, y_val)],
    categorical_feature=[c for c in cat_cols if c in selected_features],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.055640 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7020
[LightGBM] [Info] Number of data points in the train set: 303030, number of used features: 40
[LightGBM] [Info] Start training from score 7679.899902
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 2108.98
[200]	valid_0's l1: 1374.25
[300]	valid_0's l1: 1323.69
[400]	valid_0's l1: 1308.45
[500]	valid_0's l1: 1295.71
[600]	valid_0's l1: 1291.95
[700]	valid_0's l1: 1289.43
[800]	valid_0's l1: 1286.88
[900]	valid_0's l1: 1283.92
[1000]	valid_0's l1: 1281.72
[1100]	valid_0's l1: 1279.83
[1200]	valid_0's l1: 1278.44
[1300]	valid_0's l1: 1277.55
[1400]	valid_0's l1: 1276.54
[1500]	valid_0's l1: 1275.51
[1600]	valid_0's l1: 1274.75
[1700]	valid_0's l1: 1274.18
Early stopping, best iteration is:
[1712]	val

LGBMRegressor(learning_rate=0.03, n_estimators=2000, num_leaves=64,
              objective='mae', random_state=42)

# Cell 11 - Compare: did feature selection actually help?


In [21]:
val_pred_sel = model_sel.predict(X_val_sel)
wmae_sel = wmae(y_val, val_pred_sel, val['IsHoliday'])

print('Baseline (52wk):        ', baseline_score)
print('LightGBM all features:  ', wmae_all)
print('LightGBM selected feats:', wmae_sel)

best_model = model_sel if wmae_sel <= wmae_all else model_all
best_wmae = min(wmae_sel, wmae_all)
print('\nBest LightGBM variant WMAE:', best_wmae)

Baseline (52wk):         1789.9133525504185
LightGBM all features:   1301.4343028455785
LightGBM selected feats: 1302.0642579663931

Best LightGBM variant WMAE: 1301.4343028455785


# Cell 12 - Holiday vs non-holiday breakdown for the best model


In [22]:
# Worth checking separately since holidays are weighted 5x in scoring -
# a model can look fine overall but bomb on the weeks that matter most.
best_pred = val_pred_sel if best_model is model_sel else val_pred_all
is_hol = val['IsHoliday'].values

print('Holiday WMAE:    ', wmae(y_val[is_hol], best_pred[is_hol], val['IsHoliday'][is_hol]))
print('Non-holiday WMAE:', wmae(y_val[~is_hol], best_pred[~is_hol], val['IsHoliday'][~is_hol]))

Holiday WMAE:     1449.4538686533454
Non-holiday WMAE: 1262.2049694719037


# Cell 13 - Save model locally (do this regardless of whether you log to W&B yet)


In [24]:
import os
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/lightgbm_best.pkl')
print('Saved.')

Saved.


---
# Production pipeline

The requirement is that the saved pipeline runs on the **raw, un-preprocessed test
set**. The model above was trained on `data/*_processed.pkl`, so the whole of
`feature_engineering.ipynb` has to be reachable from inside the pipeline object.

`WalmartFeatureEngineer` below is that notebook re-expressed as a scikit-learn
transformer. It changes nothing about the features: it learns the train history,
rolling tails and median aggregates in `fit`, and rebuilds the same 53 columns in
`transform`. The final pipeline is

    raw rows -> WalmartFeatureEngineer -> CategoricalCaster -> FeatureSelector -> model

In [ ]:
# ============================================================
# Preprocessing pipeline components
# ============================================================
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
LAG_WEEKS = [1, 2, 4, 13, 26, 51, 52, 53]
ROLL_COLS = ['rolling_mean_4', 'rolling_median_8', 'rolling_mean_13', 'rolling_mean_26']

SUPERBOWL = ['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08']
LABORDAY = ['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06']
THANKSGIVING = ['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29']
CHRISTMAS = ['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27']


def holiday_type_of(date):
    d = date.strftime('%Y-%m-%d')
    if d in SUPERBOWL:
        return 'SuperBowl'
    if d in LABORDAY:
        return 'LaborDay'
    if d in THANKSGIVING:
        return 'Thanksgiving'
    if d in CHRISTMAS:
        return 'Christmas'
    return 'Normal'


class WalmartFeatureEngineer(BaseEstimator, TransformerMixin):
    """feature_engineering.ipynb, expressed as a transformer.

    fit(X, y)   X = raw train rows (Store, Dept, Date, IsHoliday), y = Weekly_Sales.
                Learns the sales history, the rolling-stat tail per Store-Dept and
                the median aggregates - all from train only.
    transform(X) X = raw rows (the Kaggle test set). Rebuilds the feature frame the
                model was trained on. Lags and rolling stats can only look back into
                the train history, exactly as the original notebook does for test.
    """

    def __init__(self, features, stores, lag_weeks=tuple(LAG_WEEKS)):
        self.features = features
        self.stores = stores
        self.lag_weeks = lag_weeks

    def fit(self, X, y=None):
        raw = X.copy()
        raw['Date'] = pd.to_datetime(raw['Date'])
        if y is not None:
            raw['Weekly_Sales'] = np.asarray(y)
        raw = raw.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        self.history_ = raw[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
        self.train_keys_ = raw[['Store', 'Date']].drop_duplicates()

        rs = self.history_.copy()
        g = rs.groupby(['Store', 'Dept'])['Weekly_Sales']
        rs['rolling_mean_4'] = g.shift(1).rolling(4).mean().reset_index(drop=True)
        rs['rolling_median_8'] = g.shift(1).rolling(8).median().reset_index(drop=True)
        rs['rolling_mean_13'] = g.shift(1).rolling(13).mean().reset_index(drop=True)
        rs['rolling_mean_26'] = g.shift(1).rolling(26).mean().reset_index(drop=True)

        # test is in the future, so every test row gets the last rolling value
        # available for its Store-Dept
        self.last_roll_ = (rs.sort_values('Date')
                             .groupby(['Store', 'Dept'])[ROLL_COLS]
                             .last().reset_index())

        self.dept_median_ = raw.groupby('Dept')['Weekly_Sales'].median().rename('dept_median_sales')
        self.store_median_ = raw.groupby('Store')['Weekly_Sales'].median().rename('store_median_sales')
        self.storedept_median_ = (raw.groupby(['Store', 'Dept'])['Weekly_Sales']
                                     .median().rename('storedept_median_sales'))
        return self

    def _macro(self, test_keys):
        # CPI / Unemployment are forward-filled per Store across train + test, so a
        # test week inherits the last value observed during training.
        combined = pd.concat([self.train_keys_, test_keys], ignore_index=True).drop_duplicates()
        combined = combined.merge(self.features[['Store', 'Date', 'CPI', 'Unemployment']],
                                  on=['Store', 'Date'], how='left')
        combined = combined.sort_values(['Store', 'Date'])
        for col in ['CPI', 'Unemployment']:
            combined[f'{col}_missing'] = combined[col].isna().astype('int8')
            combined[col] = combined.groupby('Store')[col].ffill()
        return combined

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        df = (df.merge(self.features, on=['Store', 'Date', 'IsHoliday'],
                       how='left', validate='many_to_one')
                .merge(self.stores, on='Store', how='left', validate='many_to_one'))

        for col in MARKDOWN_COLS:
            df[f'{col}_missing'] = df[col].isna().astype('int8')
            df[col] = df[col].fillna(0)
        df['markdown_total'] = df[MARKDOWN_COLS].sum(axis=1)
        df['markdown_count'] = (df[MARKDOWN_COLS] > 0).sum(axis=1)
        df['has_any_markdown'] = (df['markdown_total'] > 0).astype('int8')

        macro = self._macro(df[['Store', 'Date']].drop_duplicates())
        df = df.drop(columns=['CPI', 'Unemployment']).merge(macro, on=['Store', 'Date'], how='left')

        df['holiday_type'] = df['Date'].apply(holiday_type_of)

        df['year'] = df['Date'].dt.year
        df['month'] = df['Date'].dt.month
        df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
        df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
        df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)

        for w in self.lag_weeks:
            shifted = self.history_.copy()
            shifted['Date'] = shifted['Date'] + pd.Timedelta(weeks=w)
            shifted = shifted.rename(columns={'Weekly_Sales': f'lag_{w}'})
            df = df.merge(shifted, on=['Store', 'Dept', 'Date'], how='left')

        df = df.merge(self.last_roll_, on=['Store', 'Dept'], how='left')
        # A clean run of feature_engineering.ipynb produces plain rolling column
        # names. Re-running its merge cell in the same kernel produces _x / _y
        # duplicates instead. Emit both spellings so the pipeline matches either
        # schema; FeatureSelector keeps only the columns the model was trained on.
        for c in ROLL_COLS:
            df[f'{c}_x'] = df[c]
            df[f'{c}_y'] = df[c]

        df['sales_change_1'] = df['lag_1'] - df['lag_2']
        df['sales_per_store_size'] = df['lag_52'] / df['Size']
        df['markdown_per_store_size'] = df['markdown_total'] / df['Size']

        df = (df.merge(self.dept_median_, on='Dept', how='left')
                .merge(self.store_median_, on='Store', how='left')
                .merge(self.storedept_median_, on=['Store', 'Dept'], how='left'))
        return df


class CategoricalCaster(BaseEstimator, TransformerMixin):
    """Cast the categorical columns to pandas 'category' dtype."""

    def __init__(self, cat_cols):
        self.cat_cols = cat_cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cat_cols:
            if c in X.columns:
                X[c] = X[c].astype('category')
        return X


class FeatureSelector(BaseEstimator, TransformerMixin):
    """Keep the model's features, in the order it was trained on."""

    def __init__(self, features):
        self.features = features

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.features]


def build_full_pipeline(fitted_model, model_features, feature_engineer):
    # Every transformer is stateless or already fitted, so nothing is retrained here.
    return Pipeline([
        ('feature_engineering', feature_engineer),
        ('cast_categoricals', CategoricalCaster(cat_cols)),
        ('select_features', FeatureSelector(model_features)),
        ('model', fitted_model),
    ])

In [ ]:
# ============================================================
# Fit the feature engineer on the RAW training data
# ============================================================
train_raw = pd.read_csv('data/train.csv', parse_dates=['Date'])
test_raw = pd.read_csv('data/test.csv', parse_dates=['Date'])
features_raw = pd.read_csv('data/features.csv', parse_dates=['Date'])
stores_raw = pd.read_csv('data/stores.csv')

feature_engineer = WalmartFeatureEngineer(features_raw, stores_raw).fit(
    train_raw.drop(columns='Weekly_Sales'),
    train_raw['Weekly_Sales'],
)
print('Feature engineer fitted on', len(train_raw), 'raw training rows.')

In [ ]:
# ============================================================
# The winning LightGBM variant, wrapped as a raw-input pipeline
# ============================================================
best_features = feature_cols if best_model is model_all else selected_features
best_variant = 'all_features' if best_model is model_all else 'selected_features'
final_pipeline = build_full_pipeline(best_model, best_features, feature_engineer)
print('Best variant:', best_variant, '|', len(best_features), 'features')

In [ ]:
# ============================================================
# Sanity check: raw -> pipeline must reproduce the processed-data predictions
# ============================================================
_ref = test.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
_ref_pred = best_model.predict(_ref[best_features])
_pipe_pred = final_pipeline.predict(test_raw)

pipeline_max_abs_diff = float(np.abs(_ref_pred - _pipe_pred).max())
print('max |pipeline(raw) - model(processed)| =', pipeline_max_abs_diff)
assert pipeline_max_abs_diff < 1e-6, 'pipeline does not reproduce the trained model'

In [ ]:
# ============================================================
# MLflow - one experiment per architecture, one run per stage
# ============================================================
import os
import tempfile

import mlflow.sklearn

PREPROCESSING_STEPS = [
    {'step': 'load_raw', 'detail': 'read train/test/features/stores, parse Date, sort by Store-Dept-Date'},
    {'step': 'merge', 'detail': "features on ['Store','Date','IsHoliday'] + stores on ['Store']"},
    {'step': 'markdown_missing_indicators',
     'detail': 'MarkDown*_missing flags, fillna(0), markdown_total, markdown_count, has_any_markdown'},
    {'step': 'macro_forward_fill', 'detail': 'per-Store ffill of CPI / Unemployment across train+test'},
    {'step': 'holiday_type', 'detail': 'SuperBowl / LaborDay / Thanksgiving / Christmas / Normal'},
    {'step': 'calendar_features', 'detail': 'year, month, week_of_year, week_sin, week_cos'},
    {'step': 'exact_date_lags', 'detail': 'date-shifted merges; test looks back into train history only'},
    {'step': 'rolling_statistics', 'detail': 'mean_4 / median_8 / mean_13 / mean_26; test gets the last train value'},
    {'step': 'trend_and_size_features', 'detail': 'sales_change_1, sales_per_store_size, markdown_per_store_size'},
    {'step': 'fallback_aggregates', 'detail': 'dept / store / store-dept median sales, from train only'},
]




# ---------- run 1: cleaning / preprocessing ----------
with mlflow.start_run(run_name='LightGBM_Cleaning'):
    mlflow.set_tag('stage', 'cleaning')
    mlflow.log_params({
        'markdown_cols': MARKDOWN_COLS,
        'lag_weeks': LAG_WEEKS,
        'rolling_windows': [4, 8, 13, 26],
        'forward_filled_cols': ['CPI', 'Unemployment'],
        'merge_validate': 'many_to_one',
        'n_preprocessing_steps': len(PREPROCESSING_STEPS),
    })
    mlflow.log_metrics({
        'train_rows': train.shape[0],
        'train_cols': train.shape[1],
        'test_rows': test.shape[0],
        'test_cols': test.shape[1],
        'train_missing_cells': int(train.isna().sum().sum()),
        'n_stores': int(train['Store'].nunique()),
        'n_depts': int(train['Dept'].nunique()),
        'n_store_dept_pairs': int(train[['Store', 'Dept']].drop_duplicates().shape[0]),
    })
    mlflow.log_dict({'steps': PREPROCESSING_STEPS}, 'preprocessing/pipeline.json')
    mlflow.log_dict({'columns': train.columns.tolist()}, 'preprocessing/processed_columns.json')
    mlflow.log_input(
        mlflow.data.from_pandas(train, name='train_processed', targets='Weekly_Sales'),
        context='training',
    )

# ---------- run 2: baseline ----------
with mlflow.start_run(run_name='LightGBM_Baseline'):
    mlflow.set_tag('stage', 'baseline')
    mlflow.log_param('strategy', 'lag_52, falling back to storedept_median_sales')
    mlflow.log_metric('val_wmae', baseline_score)

# ---------- run 3: feature selection ----------
with mlflow.start_run(run_name='LightGBM_Feature_Selection'):
    mlflow.set_tag('stage', 'feature_selection')
    mlflow.log_params(model_all.get_params())
    mlflow.log_params({
        'selection_rule': 'importance > 0 on the all-features model',
        'n_features_before': len(feature_cols),
        'n_features_after': len(selected_features),
    })
    mlflow.log_metrics({
        'val_wmae_all_features': wmae_all,
        'val_wmae_selected_features': wmae_sel,
        'n_features_dropped': len(feature_cols) - len(selected_features),
        'best_iteration': model_all.best_iteration_,
    })
    mlflow.log_dict({'features': feature_cols}, 'features/all_features.json')
    mlflow.log_dict({'features': selected_features}, 'features/selected_features.json')
    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'feature_importances.csv')
        importances.to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='features')

# ---------- run 4: validation (chronological holdout) ----------
with mlflow.start_run(run_name='LightGBM_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.log_params({
        'scheme': 'chronological holdout (no k-fold: the target is a time series)',
        'val_weeks': 39,
        'fit_end': str(fit['Date'].max().date()),
        'val_start': str(val['Date'].min().date()),
        'val_end': str(val['Date'].max().date()),
        'fit_rows': len(fit),
        'val_rows': len(val),
    })
    mlflow.log_metrics({
        'baseline_wmae': baseline_score,
        'wmae_all_features': wmae_all,
        'wmae_selected_features': wmae_sel,
        'best_wmae': best_wmae,
        'best_wmae_holiday': wmae(y_val[is_hol], best_pred[is_hol], val['IsHoliday'][is_hol]),
        'best_wmae_non_holiday': wmae(y_val[~is_hol], best_pred[~is_hol], val['IsHoliday'][~is_hol]),
        'improvement_over_baseline': baseline_score - best_wmae,
    })

# ---------- run 5: final model + full raw-input pipeline ----------
with mlflow.start_run(run_name='LightGBM_Final_Model') as final_run:
    mlflow.set_tags({'stage': 'final_model', 'algorithm': 'lightgbm', 'variant': best_variant})
    mlflow.log_params(best_model.get_params())
    mlflow.log_param('n_features', len(best_features))
    mlflow.log_metrics({
        'final_wmae': best_wmae,
        'baseline_wmae': baseline_score,
        'pipeline_vs_processed_max_abs_diff': pipeline_max_abs_diff,
    })

    model_info = mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path='pipeline',
        # cloudpickle embeds the notebook-defined transformers in the artifact;
        # skops (the new default) refuses to serialise __main__ classes
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        input_example=test_raw.head(5),
        registered_model_name=REGISTERED_MODEL_NAME,
    )
    mlflow.log_artifact('models/lightgbm_best.pkl', artifact_path='estimator')
    print('LightGBM final run:', final_run.info.run_id)